# 泛化与校准：训练拟合好，不代表新数据好

分两部分：带噪曲线上的模型容量；人工过度自信概率的温度校准。数据与模型分数均为教学构造。先读[机器学习](../01-concepts/machine-learning/README.md)与[可信 AI](../01-concepts/trustworthy-ai/README.md)，函数见[源码](../05-code/foundations_core.py)。

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
here = Path.cwd().resolve()
repo = next(p for p in [here, *here.parents] if (p / '10-Knowledge').is_dir())
sys.path.insert(0, str(repo / '10-Knowledge' / '01-ai-foundations' / '05-code'))
from foundations_core import *
np.set_printoptions(precision=6, suppress=True)
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
print('Data: synthetic teaching examples; no model download or API call')

Python: 3.12.13 NumPy: 2.5.2
Data: synthetic teaching examples; no model download or API call


## 1. 同一训练集、独立测试点

真函数为sin(3x)，训练标签叠加高斯噪声。比较固定的三种配置：3阶、15阶、15阶＋岭惩罚。它们是为观察机制预先选定的教学配置，没有根据测试结果调参。测试误差对无噪声真函数计算，因此衡量函数恢复而非含观测噪声的预测误差。

In [2]:
rng = np.random.default_rng(7)
x_train = np.sort(rng.uniform(-1, 1, 18))
y_train = np.sin(3*x_train) + rng.normal(0,.15,18)
x_test = np.linspace(-1,1,401)
y_test = np.sin(3*x_test)
results = []
for degree, lam in [(3,0.),(15,0.),(15,.01)]:
    train_design = np.vander(x_train, degree+1, increasing=True)
    test_design = np.vander(x_test, degree+1, increasing=True)
    weights = ridge_fit(train_design,y_train,lam)
    train_mse = float(np.mean((train_design@weights-y_train)**2))
    test_mse = float(np.mean((test_design@weights-y_test)**2))
    results.append((degree,lam,train_mse,test_mse))
print('degree, lambda, train MSE, independent-grid MSE')
for row in results: print(row)
assert results[1][2] < results[0][2]
assert results[1][3] > results[0][3]
assert results[2][3] < results[1][3]

degree, lambda, train MSE, independent-grid MSE
(3, 0.0, 0.013950661655255384, 0.03288369575301072)
(15, 0.0, 0.0030065241376073756, 3175103.106445845)
(15, 0.01, 0.014514152273231063, 0.08718917848206634)


高阶能更贴近训练点，但外插到训练覆盖薄弱的区域时大幅波动。正则化抑制系数，不保证在任何随机种子和噪声强度下都优于低阶模型。试验报告保留所有配置结果，不只展示最好的一项。

## 2. 校准集选温度，测试集只评一次

用潜在logit的sigmoid生成真实标签概率，再把输出logit人为乘3，构造“排序未变、置信度偏高”的模型。独立校准集用于选温度。这里未训练分类器，也没有把模型自述信心当成概率。

In [3]:
rng = np.random.default_rng(17)
latent = rng.normal(size=5000)
labels = rng.binomial(1,sigmoid(latent))
logits = 3*latent
cal_z, test_z = logits[:1500], logits[1500:]
cal_y, test_y = labels[:1500], labels[1500:]
temperatures = np.linspace(.5,6.,111)
best_temperature = min(temperatures,key=lambda t:binary_nll(cal_z,cal_y,t))
before = calibration_metrics(test_z,test_y,bins=10)
after = calibration_metrics(test_z,test_y,best_temperature,bins=10)
print('temperature selected on calibration set:',best_temperature)
for name,report in [('before',before),('after',after)]:
    print(name,{k:round(v,6) for k,v in report.items() if k!='bins'})
assert before['accuracy'] == after['accuracy']
assert after['nll'] < before['nll']
assert after['brier'] < before['brier']
print('After calibration: count, accuracy, mean confidence')
for row in after['bins']: print(row)

temperature selected on calibration set: 2.85
before {'accuracy': 0.675714, 'nll': 0.779112, 'brier': 0.237881, 'ece': 0.162948}
after {'accuracy': 0.675714, 'nll': 0.600481, 'brier': 0.206885, 'ece': 0.016775}
After calibration: count, accuracy, mean confidence
(535, 0.5476635514018692, 0.5249862081893293)
(496, 0.5544354838709677, 0.5747198961570087)
(484, 0.5971074380165289, 0.6249008839311431)
(461, 0.6724511930585684, 0.6746543120276016)
(513, 0.7076023391812866, 0.7248163336008941)
(358, 0.7793296089385475, 0.7755967263248921)
(308, 0.8344155844155844, 0.82394935817897)
(225, 0.8488888888888889, 0.8742893079186604)
(104, 0.9038461538461539, 0.9188845706765509)
(16, 0.875, 0.9613433790050246)


## 3. 拒绝低置信度输入时，同时记录覆盖率

只对有把握的样本预测可以提高接受集准确率，但会放弃部分任务。下表使用已经固定的温度，不在测试集上挑“最好阈值”。

In [4]:
p = sigmoid(test_z/best_temperature)
confidence = np.maximum(p,1-p)
correct = (p>=.5)==test_y
for threshold in [.5,.65,.8]:
    accept = confidence>=threshold
    accuracy = float(correct[accept].mean()) if accept.any() else None
    print('threshold:',threshold,'coverage:',round(float(accept.mean()),4),'accepted accuracy:',accuracy)

threshold: 0.5 coverage: 1.0 accepted accuracy: 0.6757142857142857
threshold: 0.65 coverage: 0.5671 accepted accuracy: 0.7596977329974811
threshold: 0.8 coverage: 0.1866 accepted accuracy: 0.8514548238897397


## 观察与边界

温度缩放改变概率的尖锐程度，但不改变argmax，所以此处准确率不变、NLL和Brier改善。ECE依赖分箱，输出同时展示每箱样本数和准确率。数据被设计成共享一个过度自信倍数，现实分布偏移可能无法用单一温度修复。

练习：只对一个输入子群乘6，另一群乘1，比较全局温度与分群可靠性；注意不要利用测试标签拟合温度。